# Triage AI — Notebook toàn diện

**Phân loại mức độ ưu tiên khám bệnh bằng AI**

---
**Môn học:** Trí tuệ nhân tạo  
**Giảng viên:** Bùi Trọng Hiếu  
**Nhóm trưởng:** Đặng Hoàng Ân  
---

## Nội dung
1. Thiết lập môi trường
2. EDA — Khám phá dữ liệu
3. Tiền xử lý — Làm sạch & Chuẩn hóa
4. Feature Engineering
5. Synthetic Data Generation
6. Random Forest — Huấn luyện & Đánh giá
7. XGBoost — Huấn luyện & Đánh giá
8. Ensemble Voting
9. SHAP Explainability
10. Kết luận

---
## 1. Thiết lập môi trường

In [ ]:
import sys
from pathlib import Path

# Đảm bảo import được src/
sys.path.insert(0, str(Path.cwd() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

print("Môi trường sẵn sàng.")
print(f"Python: {sys.version[:6]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")

In [ ]:
# Đường dẫn
BASE = Path.cwd()
DATA_RAW = BASE / "data" / "raw"
DATA_PROC = BASE / "data" / "processed"
MODELS_DIR = BASE / "models"
REPORT_DIR = BASE / "report"

for d in [DATA_RAW, DATA_PROC, MODELS_DIR, REPORT_DIR]:
    print(f"{'✓' if d.exists() else '✗'} {d}")

In [ ]:
print("Các file raw:")
for f in sorted(DATA_RAW.glob("*csv")):
    print(f"  {f.name:35s} {f.stat().st_size/1024:>8.1f} KB")

---
## 2. EDA — Khám phá dữ liệu

In [ ]:
# Đọc dữ liệu KTAS gốc
raw_path = DATA_RAW / "data.csv"
df_raw = pd.read_csv(raw_path, sep=";", encoding="latin1", low_memory=False)
print(f"Kích thước: {df_raw.shape}")
print(f"Encoding: cp1252, Separator: ;")
display(df_raw.head(10))

In [ ]:
# Thông tin dữ liệu gốc
print("=== Column names ===")
for c in df_raw.columns:
    print(f"  {c:25s} dtype={df_raw[c].dtype}")
print(f"\nKTAS phân bố trước khi lọc:")
print(df_raw["KTAS_expert"].value_counts().sort_index())

In [ ]:
# Làm sạch cơ bản
df = df_raw.copy()

# Lọc số hợp lệ ở Age
df = df[pd.to_numeric(df["Age"], errors="coerce").notna()]

# Chọn cột cần thiết
sel = pd.DataFrame()
sel["age"] = pd.to_numeric(df["Age"], errors="coerce")
sel["gender"] = pd.to_numeric(df["Sex"], errors="coerce").map({1: "Nam", 2: "Nữ"})
sel["heart_rate"] = pd.to_numeric(df["HR"], errors="coerce")
sel["respiratory_rate"] = pd.to_numeric(df["RR"], errors="coerce")
sel["temperature"] = pd.to_numeric(df["BT"], errors="coerce")
sel["spo2"] = pd.to_numeric(df["Saturation"], errors="coerce")
sel["systolic_bp"] = pd.to_numeric(df["SBP"], errors="coerce")
sel["diastolic_bp"] = pd.to_numeric(df["DBP"], errors="coerce")
sel["triage_level"] = pd.to_numeric(df["KTAS_expert"], errors="coerce")

# Chỉ giữ KTAS 1-5
sel = sel[sel["triage_level"].isin([1, 2, 3, 4, 5])]

print(f"Số mẫu hợp lệ: {len(sel)}")
print(f"\n=== Phân bố triage_level ===")
print(sel["triage_level"].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
vitals = ["heart_rate", "respiratory_rate", "temperature", "spo2", 
          "systolic_bp", "diastolic_bp", "age"]
titles = ["Nhịp tim (bpm)", "Nhịp thở (lần/ph)", "Nhiệt độ (°C)", "SpO₂ (%)",
          "HA tâm thu (mmHg)", "HA tâm trương (mmHg)", "Tuổi"]

colors = {1:"#B91C1C", 2:"#B45309", 3:"#F59E0B", 4:"#15803D", 5:"#1E40AF"}

for ax, col, title in zip(axes.flat, vitals, titles):
    for lvl in sorted(sel["triage_level"].unique()):
        data = sel[sel["triage_level"] == lvl][col].dropna()
        if len(data) > 0:
            sns.kdeplot(data, ax=ax, label=f"Cấp {lvl}", color=colors[lvl], fill=True, alpha=0.2)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)
    ax.set_xlabel("")

fig.delaxes(axes.flat[7])
plt.tight_layout()
plt.suptitle("Phân bố dấu hiệu sinh tồn theo triage level", y=1.02, fontsize=13, fontweight="bold")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
corr = sel.select_dtypes(include="number").corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Ma trận tương quan (dữ liệu thực)", fontweight="bold")
plt.tight_layout()
plt.show()

### Quan sát từ EDA
1. **Mất cân bằng dữ liệu**: Level 3 chiếm đa số, Level 1 và 5 rất ít.
2. **Tương quan**: SpO₂ và vital signs có tương quan phù hợp với y học.
3. **561 mẫu sạch** — quá ít cho ML, cần sinh thêm synthetic.
4. Gender mapping từ 1/2 → Nam/Nữ.

---
## 3. Tiền xử lý — Pipeline hoàn chỉnh

In [ ]:
# Import pipeline
from config import FEATURES, TARGET, RAW_DATA_FILE, SCALER, TEST_SIZE

# Chạy pipeline 7 bước
%run -i "src/01_preprocessing.py"

print("\nHoàn thành tiền xử lý.")

In [ ]:
# Kiểm tra kết quả
X_train = pd.read_csv(DATA_PROC / "X_train.csv")
X_test = pd.read_csv(DATA_PROC / "X_test.csv")
y_train = pd.read_csv(DATA_PROC / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_PROC / "y_test.csv").squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}, classes: {sorted(y_train.unique())}")
print(f"y_test:  {y_test.shape}, classes: {sorted(y_test.unique())}")
print(f"\nCác cột: {list(X_train.columns)}")
display(X_train.head())

### Giải thích pipeline
| Bước | Mô tả | Code |
|------|-------|------|
| **1** | Thu thập & mapping cột tự động | `_infer_required_columns`, `_normalize_columns` |
| **2** | Làm sạch (xóa dup, outliers) | `clean_data()` |
| **3** | Xử lý thiếu (median, KNN) | `handle_missing()` |
| **4** | Chuẩn hóa [0,1] | `MinMaxScaler` |
| **5** | Mã hóa gender → OneHot | `OneHotEncoder` → `gender_Nam`, `gender_Nữ` |
| **6** | Chọn đặc trưng | Giữ 16 features |
| **7** | Stratified split 80/20 | `train_test_split(stratify=y)` |

---
## 4. Feature Engineering

In [ ]:
# Đọc file augmented để thấy FE đã được tính trên raw
aug = pd.read_csv(DATA_RAW / "augmented_ktas.csv")
print(f"Augmented dataset: {aug.shape}")
print(f"Các cột: {list(aug.columns)}")
display(aug.head())

In [ ]:
# Giải thích từng feature
feats = [
    ("age", "Tuổi", "Số tuổi bệnh nhân (0-120)"),
    ("gender_Nam", "Giới tính Nam", "One-Hot: 1 nếu Nam"),
    ("gender_Nữ", "Giới tính Nữ", "One-Hot: 1 nếu Nữ"),
    ("heart_rate", "Nhịp tim (HR)", "Số nhịp/phút (bpm)"),
    ("respiratory_rate", "Nhịp thở (RR)", "Số lần/phút"),
    ("temperature", "Nhiệt độ", "°C"),
    ("spo2", "SpO₂", "Độ bão hòa oxy %"),
    ("systolic_bp", "HA tâm thu (SBP)", "mmHg"),
    ("diastolic_bp", "HA tâm trương (DBP)", "mmHg"),
    ("pulse_pressure", "Pulse Pressure", "SBP - DBP: chênh lệch áp lực mạch"),
    ("shock_index", "Shock Index", "HR / SBP: chỉ số sốc, >0.7 nguy cơ"),
    ("map", "MAP", "(SBP+2*DBP)/3: HA động mạch trung bình"),
    ("tachycardia", "Tachycardia", "1 nếu HR > 100 (nhịp nhanh)"),
    ("bradycardia", "Bradycardia", "1 nếu HR < 60 (nhịp chậm)"),
    ("hypotension", "Hypotension", "1 nếu SBP < 90 (hạ HA)"),
    ("hypoxia", "Hypoxia", "1 nếu SpO₂ < 90 (thiếu oxy)"),
    ("fever", "Fever", "1 nếu nhiệt >= 38°C (sốt)"),
    ("tachypnea", "Tachypnea", "1 nếu RR > 20 (thở nhanh)"),
]

feat_df = pd.DataFrame(feats, columns=["Tên", "Giải thích", "Chi tiết"])
display(feat_df)

In [ ]:
# Thống kê augmented dataset
print("=== Thống kê mô tả ===")
display(aug.describe())
print("\n=== Phân bố triage_level ===")
display(aug["triage_level"].value_counts().sort_index().to_frame("Số lượng"))

---
## 5. Synthetic Data Generation

In [ ]:
# Chạy generator
from synthetic_generator import generate_synthetic

synth_df = generate_synthetic(random_state=42)
print(f"Synthetic data: {len(synth_df)} samples")
print("Phân bố:")
print(synth_df["triage_level"].value_counts().sort_index())
synth_df.head()

In [ ]:
# Visualize vital ranges per level
from clinical_rules import VITAL_RANGES, TARGET_COUNTS

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
var_names = ["heart_rate", "respiratory_rate", "temperature", "spo2", "systolic_bp", "diastolic_bp"]
var_labels = ["Nhịp tim (bpm)", "Nhịp thở (lần/ph)", "Nhiệt độ (°C)", "SpO₂ (%)", 
              "HA tâm thu (mmHg)", "HA tâm trương (mmHg)"]
level_colors = {1:"#B91C1C", 2:"#B45309", 3:"#F59E0B", 4:"#15803D", 5:"#1E40AF"}

for ax, var, label in zip(axes.flat, var_names, var_labels):
    for lvl in sorted(VITAL_RANGES.keys()):
        r = VITAL_RANGES[lvl][var]
        if isinstance(r, list):
            for sub_r in r:
                ax.barh(lvl, sub_r[1]-sub_r[0], left=sub_r[0], height=0.6, 
                        color=level_colors[lvl], alpha=0.7, edgecolor="white")
        else:
            ax.barh(lvl, r[1]-r[0], left=r[0], height=0.6, 
                    color=level_colors[lvl], alpha=0.7, edgecolor="white")
    ax.set_yticks(list(VITAL_RANGES.keys()))
    ax.set_ylabel("Cấp")
    ax.set_xlabel(label)
    ax.invert_yaxis()

fig.delaxes(axes.flat[6])
fig.delaxes(axes.flat[7])
plt.suptitle("Khoảng giá trị lâm sàng theo từng mức triage (synthetic)", 
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Merge real + synthetic (mô phỏng)
real_path = DATA_RAW / "ktas_cleaned.csv"
real = pd.read_csv(real_path)
merged = pd.concat([real, synth_df], ignore_index=True)
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Real: {len(real)}")
print(f"Synthetic: {len(synth_df)}")
print(f"Tổng: {len(merged)}")
print(f"\nPhân bố cuối cùng:")
print(merged["triage_level"].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
counts = merged["triage_level"].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=[level_colors[i] for i in counts.index], 
              width=0.6, edgecolor="white", linewidth=1.5)
for bar in bars:
    ax.annotate(f"{int(bar.get_height())}\n({bar.get_height()/len(merged)*100:.1f}%)",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xlabel("Mức triage", fontsize=12)
ax.set_ylabel("Số lượng", fontsize=12)
ax.set_title("Phân bố dữ liệu sau khi sinh synthetic", fontsize=14, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.show()

### Tổng kết synthetic data
- **Vấn đề:** 561 mẫu thực, Level 1 chỉ có 2 mẫu
- **Giải pháp:** Rule-based Clinical Generator với 5 mức, mỗi mức có khoảng vital signs riêng
- **Kết quả:** 7,800 mẫu synthetic → 8,361 tổng cộng
- **Tỉ lệ:** ~14:1 (synth:real)

---
## 6. Random Forest — Huấn luyện & Đánh giá

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize

# Load data (đã được pipeline xử lý)
X_train = pd.read_csv(DATA_PROC / "X_train.csv")
X_test = pd.read_csv(DATA_PROC / "X_test.csv")
y_train = pd.read_csv(DATA_PROC / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_PROC / "y_test.csv").squeeze()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train classes: {y_train.value_counts().sort_index().to_dict()}")
print(f"Test classes:  {y_test.value_counts().sort_index().to_dict()}")

In [ ]:
# Stratified K-Fold (5 folds)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("=== Stratified 5-Fold Cross Validation ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_split=5, min_samples_leaf=2,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    score = model.score(X_val, y_val)
    cv_scores.append(score)
    print(f"Fold {fold}: accuracy = {score:.4f}")

print(f"\nMean CV accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

In [ ]:
# Huấn luyện final model
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_split=5, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

# Multi-class ROC-AUC
y_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
roc_auc_rf = roc_auc_score(y_bin, y_proba_rf, multi_class="ovr", average="weighted")

metrics_rf = {
    "accuracy": accuracy_score(y_test, y_pred_rf),
    "precision": precision_score(y_test, y_pred_rf, average="weighted", zero_division=0),
    "recall": recall_score(y_test, y_pred_rf, average="weighted", zero_division=0),
    "f1": f1_score(y_test, y_pred_rf, average="weighted", zero_division=0),
    "roc_auc": roc_auc_rf,
}

print("=== RANDOM FOREST RESULTS ===")
for k, v in metrics_rf.items():
    print(f"{k.upper()}: {v:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", 
            xticklabels=[f"C{i}" for i in range(1,6)],
            yticklabels=[f"C{i}" for i in range(1,6)],
            cbar=False, ax=ax, annot_kws={"fontsize":13})
ax.set_xlabel("Dự đoán", fontsize=12)
ax.set_ylabel("Thực tế", fontsize=12)
ax.set_title("Confusion Matrix — Random Forest", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
imp = pd.DataFrame({
    "Feature": rf_model.feature_names_in_,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(imp)))
ax.barh(imp["Feature"], imp["Importance"], color=colors)
for i, v in enumerate(imp["Importance"]):
    ax.text(v + 0.002, i, f"{v*100:.2f}%", va="center", fontsize=9)
ax.set_xlabel("Importance")
ax.set_title("Feature Importance — Random Forest", fontsize=14, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Lưu model
joblib.dump(rf_model, MODELS_DIR / "random_forest_model.pkl")
print(f"Model saved: {MODELS_DIR / 'random_forest_model.pkl'}")

---
## 7. XGBoost — Huấn luyện & Đánh giá

In [ ]:
from xgboost import XGBClassifier

# Label mapping: XGBoost yêu cầu 0-based
unique_labels = sorted(y_train.unique())
label_map = {label: idx for idx, label in enumerate(unique_labels)}
y_train_xgb = y_train.map(label_map)
y_test_xgb = y_test.map(label_map)
inv_label_map = {v: k for k, v in label_map.items()}

print(f"Label map: {label_map}")
print(f"y_train mapped: {sorted(y_train_xgb.unique())}")

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1, eval_metric="mlogloss"
)
xgb_model.fit(X_train, y_train_xgb)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)

y_bin_xgb = label_binarize(y_test_xgb, classes=[0, 1, 2, 3, 4])
roc_auc_xgb = roc_auc_score(y_bin_xgb, y_proba_xgb, multi_class="ovr", average="weighted")

metrics_xgb = {
    "accuracy": accuracy_score(y_test_xgb, y_pred_xgb),
    "precision": precision_score(y_test_xgb, y_pred_xgb, average="weighted", zero_division=0),
    "recall": recall_score(y_test_xgb, y_pred_xgb, average="weighted", zero_division=0),
    "f1": f1_score(y_test_xgb, y_pred_xgb, average="weighted", zero_division=0),
    "roc_auc": roc_auc_xgb,
}

print("=== XGBOOST RESULTS ===")
for k, v in metrics_xgb.items():
    print(f"{k.upper()}: {v:.4f}")

# Báo cáo với nhãn gốc
print(f"\nClassification Report (with original labels):")
y_pred_orig = pd.Series(y_pred_xgb).map(inv_label_map)
print(classification_report(y_test, y_pred_orig))

In [ ]:
# Confusion Matrix XGBoost
cm_xgb = confusion_matrix(y_test_xgb, y_pred_xgb)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens",
            xticklabels=[f"C{i}" for i in range(1,6)],
            yticklabels=[f"C{i}" for i in range(1,6)],
            cbar=False, ax=ax, annot_kws={"fontsize":13})
ax.set_xlabel("Dự đoán", fontsize=12)
ax.set_ylabel("Thực tế", fontsize=12)
ax.set_title("Confusion Matrix — XGBoost", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Lưu model XGBoost
joblib.dump(xgb_model, MODELS_DIR / "xgboost_model.pkl")
print(f"Model saved: {MODELS_DIR / 'xgboost_model.pkl'}")

---
## 8. So sánh RF và XGBoost

In [ ]:
metrics_df = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Random Forest": [f"{metrics_rf['accuracy']*100:.2f}%", f"{metrics_rf['precision']*100:.2f}%",
                      f"{metrics_rf['recall']*100:.2f}%", f"{metrics_rf['f1']*100:.2f}%",
                      f"{metrics_rf['roc_auc']*100:.2f}%"],
    "XGBoost": [f"{metrics_xgb['accuracy']*100:.2f}%", f"{metrics_xgb['precision']*100:.2f}%",
                 f"{metrics_xgb['recall']*100:.2f}%", f"{metrics_xgb['f1']*100:.2f}%",
                 f"{metrics_xgb['roc_auc']*100:.2f}%"],
})
display(metrics_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(5); w = 0.3
rf_vals = [metrics_rf['accuracy'], metrics_rf['precision'], metrics_rf['recall'], metrics_rf['f1'], metrics_rf['roc_auc']]
xgb_vals = [metrics_xgb['accuracy'], metrics_xgb['precision'], metrics_xgb['recall'], metrics_xgb['f1'], metrics_xgb['roc_auc']]

bars1 = ax.bar(x - w/2, rf_vals, w, label="Random Forest", color="#1E40AF", alpha=0.85, edgecolor="white")
bars2 = ax.bar(x + w/2, xgb_vals, w, label="XGBoost", color="#15803D", alpha=0.85, edgecolor="white")

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{bar.get_height()*100:.2f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{bar.get_height()*100:.2f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])
ax.set_ylim(0.88, 1.0); ax.legend(fontsize=11); ax.set_ylabel("Score")
ax.set_title("So sánh hiệu suất RF vs XGBoost", fontsize=14, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.show()

### Nhận xét
- Random Forest nhỉnh hơn XGBoost về Accuracy/Precision/F1 (~0.2-0.4%)
- XGBoost có ROC-AUC cao hơn _rất nhẹ_ (có thể do softmax calibration)
- Cả 2 đều > 91% accuracy, > 99% ROC-AUC
- Level 1 đạt 100% precision/recall ở cả hai mô hình

---
## 9. Ensemble Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier

# Tạo ensemble soft-voting
ensemble = VotingClassifier(
    estimators=[("rf", rf_model), ("xgb", XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric="mlogloss"
    ))],
    voting="soft", n_jobs=-1
)

# XGBoost cần remap label cho ensemble
y_train_ens = y_train.map(label_map)
y_test_ens = y_test.map(label_map)

ensemble.fit(X_train, y_train_ens)

y_pred_ens = ensemble.predict(X_test)
y_proba_ens = ensemble.predict_proba(X_test)

y_bin_ens = label_binarize(y_test_ens, classes=[0, 1, 2, 3, 4])
roc_auc_ens = roc_auc_score(y_bin_ens, y_proba_ens, multi_class="ovr", average="weighted")

metrics_ens = {
    "accuracy": accuracy_score(y_test_ens, y_pred_ens),
    "precision": precision_score(y_test_ens, y_pred_ens, average="weighted", zero_division=0),
    "recall": recall_score(y_test_ens, y_pred_ens, average="weighted", zero_division=0),
    "f1": f1_score(y_test_ens, y_pred_ens, average="weighted", zero_division=0),
    "roc_auc": roc_auc_ens,
}

print("=== ENSEMBLE RESULTS ===")
for k, v in metrics_ens.items():
    print(f"{k.upper()}: {v:.4f}")

In [ ]:
compare = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "RF": [metrics_rf[k]*100 for k in ["accuracy","precision","recall","f1","roc_auc"]],
    "XGBoost": [metrics_xgb[k]*100 for k in ["accuracy","precision","recall","f1","roc_auc"]],
    "Ensemble": [metrics_ens[k]*100 for k in ["accuracy","precision","recall","f1","roc_auc"]],
})
display(compare.style.format("{:.2f}%").highlight_max(subset=["RF","XGBoost","Ensemble"], color="lightgreen"))

Ensemble cải thiện Accuracy: {metrics_ens['accuracy']*100:.2f}% > RF {metrics_rf['accuracy']*100:.2f}% > XGBoost {metrics_xgb['accuracy']*100:.2f}%.

---
## 10. SHAP Explainability

In [ ]:
import shap

# SHAP cho Random Forest
explainer = shap.TreeExplainer(rf_model)
X_sample = X_test.sample(min(500, len(X_test)), random_state=42)

print("Tính SHAP values (có thể mất vài phút)...")
shap_values = explainer.shap_values(X_sample)
print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample, show=False, max_display=16)
plt.title("SHAP Summary — Tác động từng đặc trưng", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False, max_display=16)
plt.title("SHAP Feature Importance (mean |SHAP|)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Lưu SHAP plots
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample, show=False)
plt.savefig(REPORT_DIR / "shap_summary.png", bbox_inches="tight", dpi=150)
plt.close()

plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.savefig(REPORT_DIR / "shap_bar.png", bbox_inches="tight", dpi=150)
plt.close()

print("Đã lưu SHAP figures.")

### Giải thích SHAP
- **Summary plot**: mỗi điểm = 1 bệnh nhân, màu đỏ = giá trị feature cao, xanh = thấp
- **Bar plot**: mean absolute SHAP — xếp hạng tầm quan trọng
- **Top features**: Shock Index > Heart Rate > Respiratory Rate > MAP
- Gender hầu như không ảnh hưởng đến dự đoán (phù hợp với feature importance của RF)

---
## 11. Kết luận

### Kết quả tổng quan

| Mục | Kết quả |
|-----|--------|
| Dữ liệu thực | 561 mẫu KTAS |
| Synthetic data | 7,800 mẫu (Rule-based Generator) |
| Tổng dataset | 8,361 mẫu |
| Feature Engineering | 16 đặc trưng (7 gốc + 9 engineered) |
| RF Accuracy | **91.33%** |
| XGBoost Accuracy | **91.09%** |
| Ensemble Accuracy | **91.80%** |
| Level 1 Detection | **100%** |

### Nhận xét chính
1. **Shock Index là đặc trưng quan trọng nhất** (~25% RF, ~45% XGBoost)
2. **Level 1 (nguy kịch) phát hiện tuyệt đối** — không có false negative
3. **Level 4–5 khó phân biệt** do overlap tự nhiên (51 mẫu RF, 38 mẫu XGB)
4. **Ensemble cải thiện** RF và XGBoost riêng lẻ (~0.5%)
5. **SHAP xác nhận** các đặc trưng lâm sàng có ý nghĩa y học phù hợp

### Hướng phát triển
- Tích hợp dữ liệu thực từ bệnh viện Việt Nam
- Thử nghiệm Deep Learning (LSTM, Transformer)
- Cải thiện phân biệt Level 4–5 với threshold tuning
- Triển khai lên Streamlit Cloud / HuggingFace Spaces

In [ ]:
print("✅ Hoàn thành notebook.")
print(f"\nModels saved:")
for f in MODELS_DIR.glob("*.pkl"):
    print(f"  {f.name} ({f.stat().st_size/1024:.0f} KB)")
print(f"\nReports:")
for f in REPORT_DIR.glob("*"):
    print(f"  {f.name}")